# NB04 · Semana 8 — Transformación con dbt (silver → gold)

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno**

## Objetivo

Formalizar las transformaciones como **código** con dbt: pasar de bronze a **silver** (tipado y limpieza) y a **gold** (métricas para consumo), con **tests de calidad** automatizados.

> Base de la **Entrega 2 (E2)**. Este notebook orquesta comandos dbt; el proyecto dbt vive en la carpeta `dbt/` del repositorio.

## 1. Estructura del proyecto dbt

Revisamos los modelos provistos como punto de partida. Cada grupo los adapta a su dominio.

In [ ]:
import subprocess, os, sys
DBT_DIR = '/home/jovyan/work'

_d = os.getcwd()
while not os.path.exists(os.path.join(_d, "entorno.py")):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d)
from entorno import STORAGE_OPTIONS

os.environ["AWS_ACCESS_KEY_ID"]     = STORAGE_OPTIONS["AWS_ACCESS_KEY_ID"]
os.environ["AWS_SECRET_ACCESS_KEY"] = STORAGE_OPTIONS["AWS_SECRET_ACCESS_KEY"]
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
endpoint = STORAGE_OPTIONS["AWS_ENDPOINT_URL"].replace("http://", "").replace("https://", "")

profile = f"""ingdatos:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: /home/jovyan/work/warehouse.duckdb
      extensions:
        - httpfs
        - delta
      secrets:
        - type: s3
          key_id: "{{{{ env_var('AWS_ACCESS_KEY_ID') }}}}"
          secret: "{{{{ env_var('AWS_SECRET_ACCESS_KEY') }}}}"
          endpoint: "{endpoint}"
          use_ssl: false
          url_style: path
          region: us-east-1
"""
with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(profile)
print("Entorno y profiles.yml listos. Endpoint:", endpoint)

## 2. Escribir todos los modelos, tests y config



In [ ]:
import os
W = "/home/jovyan/work"
for capa in ["bronze", "silver", "gold"]:
    os.makedirs(f"{W}/models/{capa}", exist_ok=True)
os.makedirs(f"{W}/tests", exist_ok=True)

# --- BRONZE ---
with open(f"{W}/models/bronze/brz_vatsim.sql", "w") as f:
    f.write('''-- Bronze: vista sobre la source lake.vatsim (tabla Delta en MinIO).
select *
from {{ source('lake', 'vatsim') }}
''')

# --- SILVER ---
with open(f"{W}/models/silver/stg_vatsim.sql", "w") as f:
    f.write('''-- Silver: tipado, limpieza y deduplicado del snapshot de VATSIM.
-- Grano: una fila por (cid, snapshot_ts). Reglas de NB03: unicidad + no-nulos.
with base as (
    select
        cast(cid as bigint)                    as cid,
        cast(callsign as varchar)              as callsign,
        cast(name as varchar)                  as piloto,
        cast(server as varchar)                as server,
        cast(latitude as double)               as latitude,
        cast(longitude as double)              as longitude,
        cast(altitude as integer)              as altitude_ft,
        cast(groundspeed as integer)           as groundspeed_kt,
        cast(heading as integer)               as heading,
        nullif(flight_plan_departure, '')      as origen,
        nullif(flight_plan_arrival, '')        as destino,
        nullif(flight_plan_aircraft_short, '') as aeronave,
        cast(snapshot_ts as timestamp)         as snapshot_ts
    from {{ ref('brz_vatsim') }}
    where cid is not null
      and snapshot_ts is not null
),
dedup as (
    select *,
        row_number() over (partition by cid, snapshot_ts order by snapshot_ts) as rn
    from base
)
select
    cast(cid as varchar) || '-' || cast(snapshot_ts as varchar) as snapshot_sk,
    cid, callsign, piloto, server, latitude, longitude,
    altitude_ft, groundspeed_kt, heading, origen, destino, aeronave, snapshot_ts
from dedup
where rn = 1
''')

# --- GOLD ---
with open(f"{W}/models/gold/fct_trafico_snapshot.sql", "w") as f:
    f.write('''-- Gold: tráfico por snapshot, listo para BI. Grano: una fila por snapshot_ts.
-- Sin 'order by': una tabla no preserva orden de lectura.
with por_snapshot as (
    select
        snapshot_ts,
        count(distinct cid)                               as pilotos_conectados,
        count(distinct cid) filter (where origen is null) as pilotos_sin_plan,
        count(distinct origen)                            as aeropuertos_origen,
        count(distinct destino)                           as aeropuertos_destino
    from {{ ref('stg_vatsim') }}
    group by snapshot_ts
),
con_previo as (
    select *,
        lag(pilotos_conectados) over (order by snapshot_ts) as pilotos_previo
    from por_snapshot
)
select
    snapshot_ts, pilotos_conectados, pilotos_sin_plan,
    aeropuertos_origen, aeropuertos_destino, pilotos_previo,
    case when pilotos_previo is null or pilotos_previo = 0 then null
         else round((pilotos_conectados - pilotos_previo) / cast(pilotos_previo as double) * 100, 2)
    end as var_pct_pilotos
from con_previo
''')

# --- SCHEMA (source + descripciones + tests) ---
with open(f"{W}/models/schema.yml", "w") as f:
    f.write('''version: 2

sources:
  - name: lake
    description: "Data lake del grupo (MinIO/S3). Capa bronze en formato Delta."
    meta:
      external_location: "delta_scan('s3://grupo03/bronze/vatsim')"
    tables:
      - name: vatsim
        description: "Snapshots crudos de pilotos VATSIM acumulados por la ingesta de E1."

models:
  - name: brz_vatsim
    description: "Bronze: vista cruda sobre la source lake.vatsim. No transforma."

  - name: stg_vatsim
    description: "Silver: snapshot VATSIM tipado y deduplicado. Grano: (cid, snapshot_ts)."
    columns:
      - name: snapshot_sk
        description: "Clave sustituta del grano (cid-snapshot_ts). Testea unicidad."
        tests: [not_null, unique]
      - name: cid
        description: "ID único del piloto en VATSIM."
        tests: [not_null]
      - name: callsign
        description: "Indicativo de vuelo."
      - name: piloto
        description: "Nombre del piloto declarado."
      - name: server
        description: "Servidor VATSIM."
      - name: latitude
        description: "Latitud (grados decimales)."
      - name: longitude
        description: "Longitud (grados decimales)."
      - name: altitude_ft
        description: "Altitud en pies (puede ser levemente negativa en tierra)."
      - name: groundspeed_kt
        description: "Velocidad respecto al suelo en nudos."
      - name: heading
        description: "Rumbo en grados."
      - name: origen
        description: "Aeropuerto de salida; nulo si no cargó plan de vuelo."
      - name: destino
        description: "Aeropuerto de llegada; nulo si no cargó plan de vuelo."
      - name: aeronave
        description: "Tipo de aeronave; nulo si no cargó plan."
      - name: snapshot_ts
        description: "Sello de tiempo de la foto."
        tests: [not_null]

  - name: fct_trafico_snapshot
    description: "Gold: tráfico por snapshot (pilotos y variación vs anterior). Para Metabase."
    columns:
      - name: snapshot_ts
        description: "Foto de la métrica. Grano del hecho."
        tests: [not_null, unique]
      - name: pilotos_conectados
        description: "Pilotos distintos conectados en el snapshot."
        tests: [not_null]
      - name: pilotos_sin_plan
        description: "Pilotos sin plan de vuelo cargado (origen nulo)."
      - name: aeropuertos_origen
        description: "Aeropuertos de salida distintos (ignora nulos)."
      - name: aeropuertos_destino
        description: "Aeropuertos de llegada distintos (ignora nulos)."
      - name: pilotos_previo
        description: "pilotos_conectados del snapshot anterior (lag)."
      - name: var_pct_pilotos
        description: "Variación % de pilotos vs snapshot anterior."
''')

# --- TESTS de rango (umbral de altitud ya corregido a -1500) ---
with open(f"{W}/tests/assert_latitud_valida.sql", "w") as f:
    f.write('''select snapshot_sk, latitude
from {{ ref('stg_vatsim') }}
where latitude is not null and (latitude < -90 or latitude > 90)
''')
with open(f"{W}/tests/assert_longitud_valida.sql", "w") as f:
    f.write('''select snapshot_sk, longitude
from {{ ref('stg_vatsim') }}
where longitude is not null and (longitude < -180 or longitude > 180)
''')
with open(f"{W}/tests/assert_altitud_valida.sql", "w") as f:
    f.write('''-- Altitud puede ser levemente negativa en tierra (QNH). Umbral físico: -1500 ft.
select snapshot_sk, altitude_ft
from {{ ref('stg_vatsim') }}
where altitude_ft is not null and altitude_ft < -1500
''')
# borrar el test viejo con umbral < 0 si quedó
viejo = f"{W}/tests/assert_altitud_no_negativa.sql"
if os.path.exists(viejo):
    os.remove(viejo)

# --- dbt_project.yml limpio ---
with open(f"{W}/dbt_project.yml", "w") as f:
    f.write('''name: 'proyecto_id'
version: '1.0.0'
config-version: 2
profile: 'ingdatos'

model-paths:  ["models"]
test-paths:   ["tests"]
target-path:  "target"
clean-targets: ["target", "dbt_packages"]

models:
  proyecto_id:
    bronze:
      +materialized: view
    silver:
      +materialized: view
    gold:
      +materialized: table
''')
print("Todos los archivos escritos (test de altitud con umbral -1500).")

## 3. Seeds (no aplica a VATSIM)

`dbt seed` carga los CSV semilla como tablas de partida (bronze) dentro de DuckDB.

In [ ]:
# VATSIM no usa seeds CSV: bronze es una tabla Delta en el lake, leída con
# delta_scan vía la source lake.vatsim. 'dbt seed' no corresponde.
print("Sin seeds: bronze es Delta en el lake (ver source lake.vatsim).")

In [ ]:
# Construir y testear
res = subprocess.run(
    ['dbt', 'build', '--profiles-dir', '.'],
    cwd=DBT_DIR, capture_output=True, text=True
)
print(res.stdout)
print(res.stderr)

## 4. Inspeccionar la capa gold

Consultamos el resultado final: métricas mensuales por variable, listas para serving/BI.

In [ ]:
import duckdb
con = duckdb.connect(os.path.join(DBT_DIR, 'warehouse.duckdb'))
gold = con.execute(
    "select * from fct_trafico_snapshot order by snapshot_ts desc limit 10"
).fetchdf()
con.close()
print(gold)

# Los 4 puntos fuertes 
El secret de DuckDB: la extensión delta no lee las settings s3_* ni variables de entorno, solo el secret manager. Fue lo que destrabó el acceso a MinIO.
    
View en bronze/silver, table en gold: las views se recalculan y dependen de credenciales del lake; la gold está materializada y se consulta sin tocar S3. Por eso la celda que leía silver falló sin el secret y la que leía gold no.
    
El test de altitud que “falló”: no era un bug, era la regla candidata mal planteada (los valores eran -2 a -7 ft, aviones en tierra). Ajustaste el umbral a -1500 ft con criterio físico en vez de silenciar el test. Eso es calidad de datos real.
     
Las reglas vienen del simulacro de NB03: unicidad de (cid, snapshot_ts) y no-nulos salieron de auditar un incidente de reingesta con time travel de Delta. El linaje conceptual E1→E2 es sólido.

In [ ]:
import subprocess
r = subprocess.run(
    "find /home/jovyan/work/models -name '.ipynb_checkpoints' -type d -exec rm -rf {} +",
    shell=True, capture_output=True, text=True
)
print("Checkpoints borrados.")
# verificar que quedaron solo los 4 archivos reales
print(subprocess.run(["find", "/home/jovyan/work/models", "-type", "f"],
                     capture_output=True, text=True).stdout)